In [1]:
pip install torch torchvision

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. Prepare Transforms and CIFAR-10 Dataset
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Automatically downloads the 163 MB CIFAR-10 python version
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

# 2. Define Model (Using a Pretrained ResNet18)
import torchvision.models as models
net = models.resnet18(pretrained=True)
net.fc = nn.Linear(net.fc.in_features, 10)  # CIFAR-10 has 10 classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net.to(device)

# 3. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

# 4. Quick Training Epoch Loop
for epoch in range(10):  # Increase epochs for actual training
    net.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch {epoch + 1} completed. Loss: {running_loss / len(trainloader):.3f}")

print("Finished Training. Ready to evaluate and push to GitHub!")

Epoch 1 completed. Loss: 1.108
Epoch 2 completed. Loss: 0.815
Epoch 3 completed. Loss: 0.725
Epoch 4 completed. Loss: 0.660
Epoch 5 completed. Loss: 0.609
Epoch 6 completed. Loss: 0.634
Epoch 7 completed. Loss: 0.571
Epoch 8 completed. Loss: 0.514
Epoch 9 completed. Loss: 0.505
Epoch 10 completed. Loss: 0.491
Finished Training. Ready to evaluate and push to GitHub!


In [6]:
net.eval()
correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        # Move to GPU if your training used CUDA
        images, labels = images.to('cuda'), labels.to('cuda')

        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the network on the 10,000 test images: {accuracy:.2f}%')

Accuracy of the network on the 10,000 test images: 82.45%
